# Solving a Krusell-Smith Model with Sequence-Space Jacobians in Julia

- Author: Yuxuan Zhao
- Date: 2026-06-07

This notebook demonstrates how to solve a Krusell-Smith model using sequence-space Jacobians. 

## 1) Model

Time is discrete. The economy is populated by a continuum of households with idiosyncratic productivity and a representative competitive firm.

The aggregate exogenous object is the TFP path

$$
Z=(Z_0,\ldots,Z_{T-1}).
$$

The path is known at date 0, so the transition is solved under perfect foresight with respect to aggregate shocks. Households still face idiosyncratic income risk.

Idiosyncratic productivity $e_t$ follows a finite-state Markov process on grid $\mathcal E$ with transition matrix $\Pi$:

$$
\Pr(e_{t+1}=e'\mid e_t=e)=\Pi(e'\mid e).
$$

The process is independent across households, so there is no aggregate uncertainty coming from idiosyncratic shocks.

The economy starts from the steady state at date 0. The initial household distribution is $D_0=D_{ss}$ and current aggregate capital is $K_0=K_{ss}$. The finite transition is computed for dates $t=0,\ldots,T-1$. The terminal conditions are

$$
Z_t=Z_{ss}\quad \text{for }t\geq T,
\qquad
K_t=K_{ss}\quad \text{for }t\geq T+1.
$$


Equivalently, the household terminal value at date $T$ is approximated by the steady-state continuation value.

Given a exogeneous path $\{Z_t\}_{t=0}^{T-1}$ and initial & terminal conditions, the perfect-foresight transition equilibrium consists of prices $\{r_t,w_t\}_{t=0}^{T-1}$, household distributions $\{D_t\}_{t=0}^{T-1}$, household policy function and value function $\{g_t,V_t\}_{t=0}^{T-1}$, aggregate quantities $\{C_t,A_{t+1},K_{t+1},Y_t\}_{t=0}^{T-1}$, and the firm's production plan such that:

### Household problem

For $t \geq T$, we have $r_t=r_{ss}$ and $w_t=w_{ss}$ for $t\geq T$, so the household problem is stationary:
$$
\begin{aligned}
V_t(e_t,a_t)=V_{ss}(e_t,a_t) =\max_{c_t,a_{t+1}}\left\{
\frac{c_t^{1-1/\psi}}{1-1/\psi}
+\beta\sum_{e_{t+1}}\Pi(e_{t+1}\mid e_t)V_{ss}(e_{t+1},a_{t+1})
\right\}
\end{aligned}
$$


For $t = 0,\ldots,T-1$, given a price path $\{r_t,w_t\}_{t=0}^{T-1}$ and terminal value $V_T=V_{ss}$, household solves:

$$
\begin{aligned}
V_t(e_t,a_t)=\max_{c_t,a_{t+1}}&\left\{
\frac{c_t^{1-1/\psi}}{1-1/\psi} 
+\beta\sum_{e_{t+1}}\Pi(e_{t+1}\mid e_t)V_{t+1}(e_{t+1},a_{t+1})
\right\} \quad \text{s.t.} \\
c_t+a_{t+1} &=(1+r_t)a_t+w_te_t, \\
\qquad a_{t+1} &\geq \underline a.
\end{aligned}
$$

Given the policy $a_{t+1}=g_t(e_t,a_t)$ and consumption policy $c_t=c_t(e_t,a_t)$, the distribution evolves forward from $D_0=D_{ss}$. Aggregate asset supply and consumption are

$$
A_{t+1}=\int g_t(e,a)\,dD_t(e,a),
\qquad
C_t=\int c_t(e,a)\,dD_t(e,a).
$$

### Firm problem

At each date $t=0,\ldots,T-1$, a representative competitive firm chooses capital and effective labor inputs $(K_t^f,L_t^f)$ to maximize static profit:

$$
\max_{K_t^f,L_t^f}\; Z_t(K_t^f)^\alpha(L_t^f)^{1-\alpha}-(r_t+\delta)K_t^f-w_tL_t^f.
$$

The first-order conditions are

$$
r_t+\delta=\alpha Z_t\left(\frac{K_t^f}{L_t^f}\right)^{\alpha-1},
\qquad
w_t=(1-\alpha)Z_t\left(\frac{K_t^f}{L_t^f}\right)^\alpha.
$$

In equilibrium, the firm rents aggregate capital and aggregate effective labor:

$$
K_t^f=K_t,\qquad L_t^f=L_t.
$$

Aggregate effective labor is the cross-sectional integral of idiosyncratic productivity,

$$
L_t=\int e\,dD_t(e,a).
$$

Because idiosyncratic shocks are independent across households and the Markov process is stationary, this aggregate is constant along the transition. We normalize this effective labor scale to match the KS1998 calibration notes, where aggregate labor is about $0.2944$ in bad times and $0.3142$ in good times. Here we use

$$
L_t=L_{ss}=0.3043.
$$

With this normalization, the firm's equilibrium conditions become

$$
Y_t=Z_tK_t^\alpha L_{ss}^{1-\alpha},
$$

$$
r_t=\alpha Z_t\left(\frac{K_t}{L_{ss}}\right)^{\alpha-1}-\delta,
\qquad
w_t=(1-\alpha)Z_t\left(\frac{K_t}{L_{ss}}\right)^\alpha.
$$
### Market clearing

The asset market clears when household aggregate saving equals next-period aggregate capital:

$$
A_{t+1}=K_{t+1},\qquad t=0,\ldots,T-1.
$$

The goods market condition is

$$
Y_t=C_t+I_t,
\qquad
I_t=K_{t+1}-(1-\delta)K_t.
$$

In the code below, the SSJ unknown is the next-period capital sequence

$$
K^+=(K_1,\ldots,K_T).
$$

The current capital sequence used by the firm is therefore

$$
(K_0,K_1,\ldots,K_{T-1})=(K_{ss},K_1,\ldots,K_{T-1}).
$$

### Summary of equilibrium conditions

Given $K^+$ and $Z$, the firm block gives prices, the household block gives aggregate asset supply $A^+=(A_1,\ldots,A_T)$, and the target equation is

$$
H_t(K^+,Z)=A_{t+1}(K^+,Z)-K_{t+1}=0,
\qquad t=0,\ldots,T-1.
$$

The goods-market residual is also computed as a diagnostic, but the SSJ solve uses the asset-market target.

## 2) Julia Setup

We use helper functions from `SSJ_Function.jl`. This file contains the generic finite-difference tools, the Rouwenhorst income process, the EGM household solver, distribution iteration, steady-state calibration, and transition-path household block.

In [ ]:
using LinearAlgebra
using Plots

include("SSJ_Function.jl")

## 3) Calibration and Steady State

We use the parameter values shown below explicitly rather than relying on hidden defaults.

### Imposed structural parameters

We follow the usual macro calibration style: preference and technology parameters are imposed directly, rather than chosen to hit steady-state targets.

$$
\beta=0.96,
\qquad
\psi=1,
\qquad
\alpha=0.36,
\qquad
\delta=0.025.
$$

The idiosyncratic productivity process is approximated by a 7-state Rouwenhorst chain:

$$
\rho_e=0.966,
\qquad
\sigma_e=0.05.
$$

The aggregate technology level is normalized to $Z_{ss}=1$. For effective labor, we use the same scale as in the KS1998 calibration notes. In those notes, hours when employed are $\tilde l=0.3271$, unemployment is $u_b=0.10$ in bad times and $u_g=0.04$ in good times, so effective aggregate labor is

$$
L_b=\tilde l(1-u_b)=0.2944,
\qquad
L_g=\tilde l(1-u_g)=0.3142.
$$

Here we use the middle of this range as the steady-state labor normalization:

$$
L_{ss}=0.3043.
$$

The asset grid is a numerical choice:

$$
n_A=500,
\qquad
 a_{min}=0,
\qquad
 a_{max}=200.
$$

### Steady-state computation

Given a candidate aggregate capital level $K$, firm prices are

$$
r(K)=\alpha Z_{ss}\left(\frac{K}{L_{ss}}\right)^{\alpha-1}-\delta,
\qquad
w(K)=(1-\alpha)Z_{ss}\left(\frac{K}{L_{ss}}\right)^\alpha.
$$

At these prices, the household steady-state problem implies aggregate asset supply $A(K)$. The steady-state capital stock is the fixed point

$$
A(K_{ss})=K_{ss}.
$$

After solving this scalar equation, output and consumption are recovered from

$$
Y_{ss}=Z_{ss}K_{ss}^{\alpha}L_{ss}^{1-\alpha},
\qquad
C_{ss}=Y_{ss}-\delta K_{ss}.
$$

Thus in this version, $\beta$, $\alpha$, $\delta$, $Z_{ss}$, and the labor scale $L_{ss}$ are imposed as structural parameters. The object solved from equilibrium is $K_{ss}$, not a parameter chosen to match a target interest rate or target output level.

In [ ]:
params = KSParams(
    beta = 0.96,
    eis = 1.0,
    delta = 0.025,
    alpha = 0.36,
    Z = 1.0,
    rho_e = 0.966,
    sd_e = 0.05,
    L = 0.3043,   # effective labor scale from KS1998: about 0.2944 to 0.3142
    nE = 7,
    nA = 500,
    amin = 0.0,
    amax = 200.0,
)

ss = compute_ks_steady_state(params)

println("beta = ", ss.beta)
println("K = ", ss.K)
println("Z = ", ss.Z)
println("r = ", ss.r)
println("w = ", ss.w)
println("Y = ", ss.Y)
println("A = ", ss.A)
println("C = ", ss.C)
println("asset_mkt = ", ss.asset_mkt)
println("goods_mkt = ", ss.goods_mkt)

The household block also produces policy functions and a stationary distribution. The plot below shows the steady-state consumption policy as a function of assets for each productivity state.

In [ ]:
plot(
    ss.a_grid,
    ss.c_policy',
    xlabel="assets",
    ylabel="consumption",
    title="Steady-state consumption policy",
    label=false,
    linewidth=1.5
)

## 4) Blocks and DAG

The DAG uses one unknown sequence and one exogenous sequence:

$$
K^+=(K_1,\ldots,K_T),\qquad Z=(Z_0,\ldots,Z_{T-1}).
$$

1. **Firm block**

   Inputs: the current capital sequence and the TFP sequence,

   $$
   K^{current}=(K_0,K_1,\ldots,K_{T-1}),\qquad Z=(Z_0,\ldots,Z_{T-1}).
   $$

   Since the unknown is $K^+=(K_1,\ldots,K_T)$ and $K_0=K_{ss}$, the code constructs

   $$
   K^{current}=lag(K^+)=(K_{ss},K_1,\ldots,K_{T-1}).
   $$

   Outputs:

   $$
   r=(r_0,\ldots,r_{T-1}),\qquad w=(w_0,\ldots,w_{T-1}),\qquad Y=(Y_0,\ldots,Y_{T-1}).
   $$

2. **Household transition block**

   Inputs:

   $$
   r=(r_0,\ldots,r_{T-1}),\qquad w=(w_0,\ldots,w_{T-1}).
   $$

   The block solves household policies backward from the terminal steady state, then pushes the distribution forward from $D_0=D_{ss}$. Outputs:

   $$
   A^+=(A_1,\ldots,A_T),\qquad C=(C_0,\ldots,C_{T-1}).
   $$

3. **Asset-market block**

   The SSJ target residual is

   $$
   asset\_mkt_t=A_{t+1}-K_{t+1},
   \qquad t=0,\ldots,T-1.
   $$

The reduced sequence-space system is therefore

$$
H(K^+,Z)=A^+(K^+,Z)-K^+=0.
$$

In [ ]:
InlineSVG(ks_dag_svg())

### Implementing the DAG blocks

The code below implements the three blocks shown in the DAG. These are model-specific functions, so they are kept in the notebook rather than hidden inside `SSJ_Function.jl`.

### Firm block: from capital and TFP to prices

Input paths are the unknown next-period capital sequence $K^+=(K_1,\ldots,K_T)$ and the exogenous TFP sequence $Z=(Z_0,\ldots,Z_{T-1})$. Since firms at date $t$ use current capital, the first step is the timing conversion

$$
K^{current}=(K_0,K_1,\ldots,K_{T-1})=(K_{ss},K_1,\ldots,K_{T-1}).
$$

For each date $t=0,\ldots,T-1$, the firm block applies the static FOCs

$$
r_t=\alpha Z_t\left(\frac{K_t}{L}\right)^{\alpha-1}-\delta,
\qquad
w_t=(1-\alpha)Z_t\left(\frac{K_t}{L}\right)^\alpha,
$$

and production

$$
Y_t=Z_tK_t^\alpha L^{1-\alpha}.
$$

Output: paths $(r,w,Y)$.

In [ ]:
# Timing helper: convert next-period capital choices K^+ = (K_1,...,K_T)
# into current capital used by firms, (K_0,K_1,...,K_{T-1}).
function lag_path(x, x0)
    return vcat(x0, x[1:end-1])
end

# Firm block
# Inputs:
#   Kplus: next-period capital sequence (K_1,...,K_T)
#   Z:     TFP sequence (Z_0,...,Z_{T-1})
#   ss:    steady-state object, used for initial capital K_0
#   params: model parameters
# Outputs:
#   r, w, Y: price and output sequences for dates 0,...,T-1
function firm_paths(Kplus, Z, ss, params)
    K_current = lag_path(Kplus, ss.K)
    L = ss.L

    r = params.alpha .* Z .* (K_current ./ L).^(params.alpha - 1) .- params.delta
    w = (1 - params.alpha) .* Z .* (K_current ./ L).^params.alpha
    Y = Z .* K_current.^params.alpha .* L.^(1 - params.alpha)

    return r, w, Y
end

### Household transition block: from prices to aggregate saving

Input paths are prices $(r_0,\ldots,r_{T-1})$ and $(w_0,\ldots,w_{T-1})$. The block solves the household problem backward using terminal value $V_T=V_{ss}$:

$$
V_t(e,a)=\max_{c,a'}\left\{u(c)+\beta\sum_{e'}\Pi(e'\mid e)V_{t+1}(e',a')\right\}
$$

subject to

$$
c+a'=(1+r_t)a+w_te,
\qquad a'\geq \underline a.
$$

This gives policy functions $a'=g_t(e,a)$ and $c=c_t(e,a)$. Then the distribution is pushed forward from $D_0=D_{ss}$, and we aggregate

$$
A_{t+1}=\int g_t(e,a)dD_t(e,a),
\qquad
C_t=\int c_t(e,a)dD_t(e,a).
$$

Output: paths $(A^+,C)$.

In [ ]:
# Household transition block
# Inputs:
#   r_path, w_path: price sequences for dates 0,...,T-1
#   ss: terminal steady state and initial distribution D_0
#   params: model parameters
# Outputs:
#   A: next-period aggregate asset supply sequence (A_1,...,A_T)
#   C: aggregate consumption sequence (C_0,...,C_{T-1})
function household_transition(r_path, w_path, ss, params)
    T = length(r_path)

    # Backward pass: solve household policies from T-1 back to 0.
    # The terminal object is the steady-state marginal value V_A,T = V_A,ss.
    Va_next = ss.Va
    a_policies = Vector{Matrix{Float64}}(undef, T)
    c_policies = Vector{Matrix{Float64}}(undef, T)

    for t in T:-1:1
        Va, a_policy, c_policy = egm_step(
            Va_next, ss.a_grid, ss.e_grid, ss.Pi,
            r_path[t], w_path[t], ss.beta, params.eis
        )
        a_policies[t] = a_policy
        c_policies[t] = c_policy
        Va_next = Va
    end

    # Forward pass: start from D_0 = D_ss and aggregate decisions.
    D = ss.D
    A = zeros(T)
    C = zeros(T)

    for t in 1:T
        A[t] = aggregate_assets(D, a_policies[t])
        C[t] = aggregate_consumption(D, c_policies[t])
        D = forward_distribution(D, a_policies[t], ss.a_grid, ss.Pi)
    end

    return A, C
end

### Asset-market block: from household saving to target residual

Given $K^+$ and $Z$, the firm and household blocks imply household aggregate asset supply

$$
A^+(K^+,Z)=(A_1,\ldots,A_T).
$$

The equilibrium target is asset-market clearing at each date:

$$
H_t(K^+,Z)=A_{t+1}(K^+,Z)-K_{t+1}=0,
\qquad t=0,\ldots,T-1.
$$

The code also computes the goods-market residual

$$
goods\_mkt_t=Y_t-C_t-I_t,
\qquad I_t=K_{t+1}-(1-\delta)K_t,
$$

but this is a diagnostic, not the SSJ target.

In [ ]:
# Reduced sequence-space residual map
# Inputs:
#   Kplus: unknown path (K_1,...,K_T)
#   Z:     exogenous path (Z_0,...,Z_{T-1})
# Output:
#   named tuple containing the target residual and useful model variables
function ks_residual(Kplus, Z, ss, params)
    r, w, Y = firm_paths(Kplus, Z, ss, params)
    A, C = household_transition(r, w, ss, params)

    # Target equation: household asset supply must equal next-period capital.
    asset_mkt = A .- Kplus

    # Goods-market residual is only a diagnostic.
    K_current = lag_path(Kplus, ss.K)
    I = Kplus .- (1 - params.delta) .* K_current
    goods_mkt = Y .- C .- I

    return (; asset_mkt, goods_mkt, A, C, I, r, w, Y)
end

# Convenience map for later Jacobian calculations.
# Given Kplus and Z in levels, return one model variable in levels.
function ks_variable_levels(varname, Kplus, Z, ss, params)
    out = ks_residual(Kplus, Z, ss, params)

    if varname == :K
        return Kplus
    elseif varname == :Z
        return Z
    else
        return getproperty(out, varname)
    end
end

In [ ]:
T = 80
Kplus_ss_path = fill(ss.K, T)
Z_ss_path = fill(ss.Z, T)

out_ss = ks_residual(Kplus_ss_path, Z_ss_path, ss, params)
println("Maximum steady-state asset-market residual: ", maximum(abs.(out_ss.asset_mkt)))
println("Maximum steady-state goods-market residual: ", maximum(abs.(out_ss.goods_mkt)))

## 5) Jacobians and Linearized Dynamics

Around the steady state, the residual map is written in terms of the next-period capital path $K^+=(K_1,\ldots,K_T)$:

$$
H(K^+_{ss}+dK^+,Z_{ss}+dZ)\approx H(K^+_{ss},Z_{ss})+H_KdK^++H_ZdZ.
$$

Since $H(K^+_{ss},Z_{ss})=0$, the linearized equilibrium condition is

$$
H_KdK^++H_ZdZ=0.
$$

Thus the general-equilibrium Jacobian for the next-period capital path is

$$
G^{K^+,Z}=\frac{dK^+}{dZ}=-H_K^{-1}H_Z.
$$

The code below computes $H_K$ and $H_Z$ by finite differences of the full residual map. Conceptually, this is the same object produced by the Python SSJ package after composing the firm and household Jacobians along the DAG.

In [ ]:
H_K = finite_difference_jacobian_at(K -> ks_residual(K, Z_ss_path, ss, params).asset_mkt, Kplus_ss_path; h=1e-5)
H_Z = finite_difference_jacobian_at(Z -> ks_residual(Kplus_ss_path, Z, ss, params).asset_mkt, Z_ss_path; h=1e-5)

G_KZ = -H_K \ H_Z

println("size(H_K) = ", size(H_K))
println("size(H_Z) = ", size(H_Z))
println("size(G_KZ) = ", size(G_KZ))

Once $G^{K^+,Z}$ is known, other aggregate variables are recovered by the chain rule. For a variable $X=X(K^+,Z)$,

$$
G^{X,Z}=X_KG^{K^+,Z}+X_Z.
$$

This is the same logic as in the RBC notebook, but the map $X(K^+,Z)$ now includes the heterogeneous-agent household transition block.

In [ ]:
function ks_variable_jacobian(varname, G_KZ, ss, params; T=80, h=1e-5)
    Kss = fill(ss.K, T)
    Zss = fill(ss.Z, T)
    X_K = finite_difference_jacobian_at(K -> ks_variable_levels(varname, K, Zss, ss, params), Kss; h=h)
    X_Z = finite_difference_jacobian_at(Z -> ks_variable_levels(varname, Kss, Z, ss, params), Zss; h=h)
    return X_K * G_KZ + X_Z
end

G_rZ = ks_variable_jacobian(:r, G_KZ, ss, params; T=T)
G_wZ = ks_variable_jacobian(:w, G_KZ, ss, params; T=T)
G_YZ = ks_variable_jacobian(:Y, G_KZ, ss, params; T=T)
G_CZ = ks_variable_jacobian(:C, G_KZ, ss, params; T=T)
G_AZ = ks_variable_jacobian(:A, G_KZ, ss, params; T=T)

println("size(G_CZ) = ", size(G_CZ))

## 6) Impulse Responses

After the Jacobians are computed, any small TFP shock path is handled by matrix multiplication. For example,

$$
dK^+=G^{K^+,Z}dZ,\qquad dC=G^{C,Z}dZ.
$$

We use a 1 percent AR(1)-style TFP shock,

$$
dZ_t=0.01 Z_{ss}\rho^t,
$$

and compare responses for several values of $\rho$.

In [ ]:
rhos = [0.2, 0.4, 0.6, 0.8, 0.9]
dZ = hcat([0.01 .* ss.Z .* (rho .^ (0:T-1)) for rho in rhos]...)

# Responses implied by the general-equilibrium Jacobians.
dr = 10_000 .* (G_rZ * dZ)
dK = 100 .* (G_KZ * dZ) ./ ss.K
dC = 100 .* (G_CZ * dZ) ./ ss.C

plot_horizon = 50
quarters = 0:plot_horizon-1
labels = ["rho=$(rho)" for rho in rhos]

dZ_percent = 100 .* dZ ./ ss.Z

p0 = plot(
    quarters,
    dZ_percent[1:plot_horizon, :],
    label=reshape(labels, 1, :),
    linewidth=2,
    xlabel="quarters",
    ylabel="percent deviation",
    title="TFP shock paths"
)

p1 = plot(
    quarters,
    dr[1:plot_horizon, :],
    label=reshape(labels, 1, :),
    linewidth=2,
    xlabel="quarters",
    ylabel="basis points",
    title="Interest rate responses"
)

p2 = plot(
    quarters,
    dK[1:plot_horizon, :],
    label=reshape(labels, 1, :),
    linewidth=2,
    xlabel="quarters",
    ylabel="percent deviation",
    title="Capital responses"
)

plot(p0, p1, p2, layout=(1, 3), size=(1200, 350))

## 7) News Shocks

A news shock is simply a TFP path whose nonzero entry starts in the future. The linear response is obtained using the same Jacobian matrix. The columns of $G^{K^+,Z}$ can be interpreted as responses of the next-period capital path to one-period TFP news arriving at different dates.

In [ ]:
news_dates = [5, 10, 15, 20, 25]
dZ_news = zeros(T, length(news_dates))
for (j, date) in enumerate(news_dates)
    dZ_news[date + 1, j] = 0.01 * ss.Z
end

# Responses to one-period TFP news shocks arriving at different future dates.
dr_news = 10_000 .* (G_rZ * dZ_news)
dK_news = 100 .* (G_KZ * dZ_news) ./ ss.K
dZ_news_percent = 100 .* dZ_news ./ ss.Z
news_labels = ["news at t=$d" for d in news_dates]

p_news_z = plot(
    quarters,
    dZ_news_percent[1:plot_horizon, :],
    label=reshape(news_labels, 1, :),
    linewidth=2,
    xlabel="quarters",
    ylabel="percent deviation",
    title="TFP news paths"
)

p_news_r = plot(
    quarters,
    dr_news[1:plot_horizon, :],
    label=reshape(news_labels, 1, :),
    linewidth=2,
    xlabel="quarters",
    ylabel="basis points",
    title="Interest rate responses"
)

p_news_k = plot(
    quarters,
    dK_news[1:plot_horizon, :],
    label=reshape(news_labels, 1, :),
    linewidth=2,
    xlabel="quarters",
    ylabel="percent deviation",
    title="Capital responses"
)

plot(p_news_z, p_news_r, p_news_k, layout=(1, 3), size=(1200, 350))

## 8) Nonlinear Perfect-Foresight Response

Everything above used a first-order approximation around the steady state. For a small TFP path $dZ$, we used the Jacobian formula

$$
dK^+=G^{K^+,Z}dZ.
$$

This is fast, but it is only a local linear approximation. In this section we instead solve the full nonlinear perfect-foresight transition problem for a larger TFP shock.

Given a level path

$$
Z=(Z_0,\ldots,Z_{T-1}),
$$

the unknown is still the next-period capital path

$$
K^+=(K_1,\ldots,K_T).
$$

For any candidate $K^+$, the residual map does the full nonlinear calculation:

1. firm block: $(K^+,Z)\rightarrow (r,w,Y)$,
2. household transition block: $(r,w)\rightarrow (A^+,C)$,
3. asset-market residual:

$$
H(K^+,Z)=A^+(K^+,Z)-K^+.
$$

The nonlinear equilibrium condition is

$$
H(K^+,Z)=0.
$$

The code below solves this equation by a quasi-Newton method. Starting from the steady-state path $K^+_0=(K_{ss},\ldots,K_{ss})$, each iteration evaluates the nonlinear residual exactly, but uses the steady-state Jacobian $H_K$ as an approximate derivative:

$$
K^+_{n+1}=K^+_n-H_K^{-1}H(K^+_n,Z).
$$

So this is not just matrix-multiplying the linear response. The residual $H(K^+_n,Z)$ is recomputed from the full nonlinear firm and household blocks at every iteration. Finally, we compare the nonlinear response with the linear approximation from the Jacobian:

$$
K^+_{linear}=K^+_{ss}+G^{K^+,Z}dZ.
$$

If the shock is small, the two responses should be close. If the shock is large, the difference shows the importance of nonlinearities.

In [ ]:
function solve_ks_nonlinear(Z_level, ss, params, H_K; tol=1e-8, max_iter=30)
    T = length(Z_level)
    K = fill(ss.K, T)
    out = ks_residual(K, Z_level, ss, params)

    for it in 1:max_iter
        err = maximum(abs.(out.asset_mkt))
        if err < tol
            return K, out, it
        end
        K += -(H_K \ out.asset_mkt)
        out = ks_residual(K, Z_level, ss, params)
    end
    error("Nonlinear solver did not converge.")
end

big_dZ = 0.10 .* ss.Z .* (0.8 .^ (0:T-1))
Z_big = fill(ss.Z, T) + big_dZ

K_nonlin, td_nonlin, iters = solve_ks_nonlinear(Z_big, ss, params, H_K)
K_lin = fill(ss.K, T) + G_KZ * big_dZ
r_lin = ss.r .+ G_rZ * big_dZ

println("Nonlinear iterations: ", iters)
println("Maximum nonlinear asset-market residual: ", maximum(abs.(td_nonlin.asset_mkt)))

p_nonlin = plot(
    quarters,
    10_000 .* (td_nonlin.r[1:plot_horizon] .- ss.r),
    label="nonlinear",
    linewidth=2.5,
    xlabel="quarters",
    ylabel="basis points",
    title="Interest rate response to a 10% TFP shock"
)
plot!(
    p_nonlin,
    quarters,
    10_000 .* (r_lin[1:plot_horizon] .- ss.r),
    label="linear",
    linewidth=2.5,
    linestyle=:dash
)